# A Simple RAG Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:

# Run once when setting up the project
#!git clone 
#%cd simple-rag-project
#!git pull

# Run once if packages are missing
# !pip install -r requirements.txt

  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached jupyter_lsp-2.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached jupyter_events-0.12.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached jupyter_server_terminals-0.5.4-py3-none-any.whl.metadata (5.9 kB)
  Using cached send2trash-2.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached terminado-0.18.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached rfc3339_validator-0.1.4-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached rfc3986_validator-0.1.1-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached rfc3987_syntax-1.1.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached

In [9]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv


In [17]:
# ==============================
# 2. Load environment variables
# ==============================

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to .env locally, or to Colab Secrets."
    )


# ==============================
# 3. Set up the clients
# ==============================

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("OPENAI_API_BASE", "https://api.openai.com/v1"),
)

chat_model = os.getenv("MODEL_ECONOMY", "gpt-4o-mini")

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

print("Chat model:", chat_model)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chat model: gpt-4o-mini
Embedding dimension: 768


/var/folders/v2/lnk8lqt55fdgnft9gn26gclr0000gn/T/ipykernel_39066/401913836.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [11]:
# ==============================
# 3. Load knowledge base
# ==============================

with open("./data/knowledge.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Each paragraph becomes one chunk
chunks = [p.strip() for p in text.split("\n\n") if p.strip()]

print("Chunks:")
for c in chunks:
    print("-", c)


Chunks:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.
- The Amazon rainforest is the largest tropical rainforest in the world. It spans multiple countries in South America and contains immense biodiversity.
- The Pacific Ocean is the largest ocean on Earth. It covers more than 30% of the planet’s surface.


In [12]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)


FAISS index size: 4


In [13]:
# ==============================
# 6. User question
# ==============================

#question = "Which ocean is the largest on Earth?"
question = "Is Great Wall of China visible from the Moon?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

# You can print out the embedding just in case you want to see how it looks
#print("Question Embedding:", question_embedding)



Question: Is Great Wall of China visible from the Moon?


In [14]:
# ==============================
# 8. Retrieve top 2 chunks
# ==============================

k = 2

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)


Retrieved context:
- Humans only use 2% of their brains, the Great Wall of China is visible from the Moon, drinking eight liters of water a day is essential for survival, and lightning never strikes the same place twice.
- Mount Everest is the tallest mountain on Earth. Its peak is 8848 meters above sea level and it is part of the Himalayas.


In [15]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

prompt = f"""
# Instructions
Answer the following question in one short response.
Use only the context provided, even if it contains false facts.
Say "There is no information awailable" if there is no answer in the context

# Context:
{context}

# Question:
{question}

# Answer:
"""

#print("Prompt:", prompt)


In [16]:
# ==============================
# 10. Call GPT-4o to generate the answer
# ==============================

response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)


LLM Answer:
Yes, the Great Wall of China is visible from the Moon.
